# 01 Feature and Data Prep

This notebook builds the unified modeling dataset, defines hit labels, harmonizes compound identifiers, and prepares feature sources.

In [1]:
import pandas as pd
from pathlib import Path

from gbm_pipeline import (
    CAT_COLS,
    define_hit_label,
    load_ctd_mapping,
    normalize_name,
)

hts_path = 'HTS_summary.csv'
df = pd.read_csv(hts_path)
print(df.shape)
df.head()

(6221, 24)


,Protocol Name,Sample ID,Structure,Sample Name,CAS#,Sample Data Type,AC50 (uM),CC-v2,Efficacy,Max Resp,...,Hit?,MeSH Terms,FDA Pharmacological Classes,MoA,PE,CS,EPC,USAN_definition,Any_category?,CTD_chemical_name
0,TRND-Glioblastoma_LOPAC_U251-primary_screen2,NCGC00015647-36,COC(=O)NC1=NC2=C(N1)C=CC(=C2)C(=O)C3=CC=CS3,Nocodazole,31430-18-9,activity,0.000708,5.0,-39.415,-58.792,...,Non-hit,Tubulin Modulators; Antineoplastic Agents,NaN,NaN,NaN,NaN,NaN,NaN,Yes,Nocodazole
1,TRND-Glioblastoma_LOPAC_U251-primary_screen2,NCGC00163700-23,CC[C@]1(O)C[C@H]2CN(CCC3=C([NH]C4=C3C=CC=C4)[C...,Vincristine sulfate,[NO STEREO] 57-22-7,activity,0.019953,5.0,-29.928,-62.381,...,Non-hit,"Antineoplastic Agents, Phytogenic; Tubulin Mod...",Vinca Alkaloid [EPC]; Vinca Alkaloids [CS],NaN,NaN,Vinca Alkaloids,Vinca Alkaloid,antihistaminics (histamine-H1 receptor antagon...,Yes,Vincristine
2,TRND-Glioblastoma_U251_NPACT-primary_screen,NCGC00386361-03,CN1CCN(CC1)C2=CC=C(OC(F)(F)F)C(=C2)NC3=NC4=C(C...,NMS-P937,1034616-18-6,activity,0.031623,5.0,-52.998,-50.661,...,Non-hit,Protein Kinase Inhibitors,NaN,NaN,NaN,NaN,NaN,serine/threonine kinase inhibitors,Yes,NMS P937
3,TRND-Glioblastoma_NPC_U251-primary_screen,NCGC00274030-12,CCN1CCN(CC1)C2=CC=C(NC3=CC(=NC=N3)N(C)C(=O)NC4...,Infigratinib,NaN,activity,0.055574,-1.2,-47.115,-18.012,...,Non-hit,Antineoplastic Agents,NaN,NaN,NaN,NaN,NaN,NaN,Yes,infigratinib
4,TRND-Glioblastoma_U251_NPACT-primary_screen,NCGC00390406-01,[Na]OC(=O)CCNC1=NC(=NC(=C1)N2CCC3=CC=CC=C3CC2)...,GSK-J1sodiumsalt,NaN,activity,0.079433,-2.4,-64.454,-6.808,...,Non-hit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN


## Define Target Label

Use the provided hit criteria to define the target label and compare against the existing `Hit?` column.

In [2]:
df = df.copy()
df['label_rule'] = define_hit_label(df)
df['label_hit_column'] = df['Hit?'].map({'Hit': 1, 'Non-hit': 0})

print(df['label_rule'].value_counts())
print(df['label_hit_column'].value_counts(dropna=False))

comparison = pd.crosstab(df['label_rule'], df['label_hit_column'], dropna=False)
comparison

label_rule
1    3464
0    2757
Name: count, dtype: int64
label_hit_column
0    3971
1    2250
Name: count, dtype: int64


label_hit_column,0,1
label_rule,,
0,2757,0
1,1214,2250


## Harmonize Chemical Names

Normalize `CTD_chemical_name` to improve join consistency with CTD annotation files.

In [3]:
df['ctd_name_norm'] = df['CTD_chemical_name'].map(normalize_name)
df['ctd_name_norm'].isna().mean()

0.33065423565343194

## Load CTD Annotation Mappings

Build chemical -> annotation mappings for pathways, genes, diseases, GO, and phenotypes.

In [4]:
ctd_pathways = load_ctd_mapping('CTD mapping/CTD_chem_pathways_enriched_1535.tsv', 'PathwayName')
ctd_genes = load_ctd_mapping('CTD mapping/CTD_chem_cgixns_1535.tsv', 'GeneSymbol')
ctd_diseases = load_ctd_mapping('CTD mapping/CTD_chem_diseases_1535.tsv', 'DiseaseID')
ctd_go = load_ctd_mapping('CTD mapping/CTD_chem_go_enriched_1535.tsv', 'GoTermID')
ctd_phenotypes = load_ctd_mapping('CTD mapping/CTD_chem_phenotypes_curated_1535.tsv', 'PhenotypeID')

print('pathways', len(ctd_pathways))
print('genes', len(ctd_genes))
print('diseases', len(ctd_diseases))
print('go', len(ctd_go))
print('phenotypes', len(ctd_phenotypes))

pathways 693
genes 1200
diseases 1339
go 711
phenotypes 923


## Save Prepared Dataset

Persist the cleaned dataset for downstream modeling notebooks.

In [5]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / 'hts_prepared.csv', index=False)

print('Saved', output_dir / 'hts_prepared.csv')

Saved outputs/hts_prepared.csv
